# 3D representation: synthetic phantom baseline

Status: NOT RUN. A small 3D autoencoder reconstructs aligned 32³ volumes. The default is explicitly synthetic ellipsoid data, not ultrasound simulation or anatomical evidence. GPU is helpful; one volume per batch limits memory. Real scans require validated geometry and a prepared NPZ manifest.

In [ ]:
from pathlib import Path
import os, sys
REPO_ROOT = Path(os.environ.get("SB_REPO_ROOT", "/CHANGE/TO/UPLOADED/REPOSITORY"))
assert (REPO_ROOT / "requirements-training.txt").is_file(), "Set REPO_ROOT to the folder containing README.md"
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# Run this only in your selected Colab/Kaggle/local training environment.
# Kaggle requires internet enabled to install/download pretrained weights.
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-training.txt"])

In [ ]:
from build.imagery.ultrasound_3d.pipeline import synthetic_dataset, train, visualize
DATA_ROOT = REPO_ROOT / "output/datasets/ultrasound/synthetic"
MANIFEST = synthetic_dataset(DATA_ROOT, count=60, seed=42)
# For real data replace BOTH paths with your approved, calibrated volume dataset.

In [ ]:
RESUME = None
report = train(MANIFEST, DATA_ROOT, epochs=10, seed=42, resume=RESUME)
report

In [ ]:
from build.integration.common import read_json
m = read_json(MANIFEST)
sample = next(s for s in m["samples"] if s["split"]=="test")
destination = REPO_ROOT/"output/visualizations/volume_reconstruction.png"
destination.parent.mkdir(parents=True,exist_ok=True)
visualize(REPO_ROOT/"output/models/model_2_ultrasound_3d", DATA_ROOT/sample["path"], destination)
from IPython.display import display, Image
display(Image(filename=str(destination)))

Compare test reconstruction MSE with the constant-mean baseline. A low MSE alone does not establish anatomy. Preserve model_2_ultrasound_3d and the generated volume metadata. DICOM and untracked freehand scan ingestion require a separately validated adapter; do not stack unrelated images.

## Download the complete model
Run only after successful training. The ZIP includes weights, metadata and evaluation. Save checkpoints separately if you want to resume later.

In [ ]:
from scripts.export_model_bundle import export
bundle = export('model_2_ultrasound_3d')
from IPython.display import display, FileLink
display(FileLink(str(bundle.relative_to(REPO_ROOT))))
print("Download this file using the notebook file browser if the link is unavailable:", bundle)
